# M3-B1 — Mission étoile : préparer la 3ᵉ source pour du RAG — SUJET

> **Bonus optionnel, non noté.** Acerox t'a transmis un corpus de **rapports
> techniques** (`rapports_techniques_2024.zip`, 5 fichiers `.md`). C'est du
> **texte non structuré** : impossible à ranger dans une table SQL. Tu vas le
> **préparer** pour une exploitation par similarité sémantique.

## 🚫 Garde-fou
> **Préparation seulement** : tu t'arrêtes à la **récupération** (retrieval).
> La génération augmentée par un LLM, c'est **M7-B2**. Pour la prédiction de
> défauts (tabulaire), un RAG **n'aide pas** : ce corpus sert l'**aide au
> diagnostic humain**. Embeddings **locaux** (aucune clé API).

## Pré-requis
```bash
# Dézippe rapports_techniques_2024.zip dans  ../data/rapports_techniques_2024/
# Décommente le bloc bonus de requirements.txt puis :
pip install chromadb sentence-transformers
```

## 1. Repérer les rapports (fourni)

In [25]:
from pathlib import Path

RAPPORTS = Path("../data/rapports_techniques_2024")
EMBED_MODEL = "all-MiniLM-L6-v2"  # Adapté au corpus ? (regarde la langue des rapports)

fichiers = sorted(RAPPORTS.glob("*.md"))
print(f"{len(fichiers)} rapports trouvés :")
for f in fichiers:
    print(" -", f.name)

5 rapports trouvés :
 - RT-2026-001_Roubaix_ligne1.md
 - RT-2026-002_Lyon_ligne2.md
 - RT-2026-003_Lyon_ligne4.md
 - RT-2026-004_Lyon_ligne4.md
 - RT-2026-005_Lyon_ligne2.md


## 2. DocumentLoader (fait main) — **TODO**

Écris une fonction qui lit un `.md` et renvoie un dict avec au moins :
`source`, `reference`, `date` (extraits de l'en-tête par regex) et `texte`.
Pas besoin de LangChain.

In [26]:
import re


def charger_rapport(path: Path) -> dict:
    contenu = path.read_text(encoding="utf-8")
    lignes = contenu.splitlines()

    entete = next((ligne for ligne in lignes if ligne.strip().startswith("> Référence")), "")

    reference_match = re.search(r"Référence\s*:\s*([^|\n]+)", entete)
    date_match = re.search(r"Date\s*:\s*([^|\n]+)", entete)
    site_match = re.search(r"Site\s*:\s*([^|\n]+)", entete)
    ligne_match = re.search(r"Ligne\s*:\s*([^|\n]+)", entete)

    corps = []
    for ligne in lignes:
        if ligne.startswith("# ") or ligne.strip().startswith("> Référence"):
            continue
        corps.append(ligne)

    texte = "\n".join(corps).strip()

    return {
        "source": path.name,
        "reference": reference_match.group(1).strip() if reference_match else None,
        "date": date_match.group(1).strip() if date_match else None,
        "site": site_match.group(1).strip() if site_match else None,
        "ligne": ligne_match.group(1).strip() if ligne_match else None,
        "texte": texte,
    }


docs = [charger_rapport(f) for f in fichiers]
docs[0]

{'source': 'RT-2026-001_Roubaix_ligne1.md',
 'reference': 'RT-2026-001',
 'date': '2026-04-01',
 'site': 'Roubaix',
 'ligne': '1',
 'texte': "## Dérive thermique ligne 1\n\nSynthèse hebdomadaire des alertes supervision.\n\nTempérature moyenne relevée à 193 °C sur 6 h, au-dessus du seuil de consigne (180 °C). Cause probable : encrassement de l'échangeur. Intervention de nettoyage planifiée, retour à la normale sous 48 h.\n\n## Suite donnée\n\n- Responsable : EMP-2424\n- Statut : en cours\n- Lien supervision : voir `capteurs_iot.csv` (2026-04-01, site Roubaix, line_id 1)."}

## 3. Segmentation (chunking) — **TODO**

Implémente **2 stratégies** et **justifie ton choix** :
- **par titre Markdown** (`##`) : chunks cohérents avec la structure ;
- **par taille fixe** (avec recouvrement) : chunks homogènes.

> C'est l'item syllabus « implémentation de techniques de segmentation ».

In [27]:
def chunk_par_titre(doc: dict) -> list[dict]:
    texte = doc["texte"]
    sections = re.split(r"(?=^##\s+)" , texte, flags=re.MULTILINE)
    chunks = []

    for index, section in enumerate(sections, start=1):
        section = section.strip()
        if not section:
            continue

        lignes = section.splitlines()
        titre = lignes[0].removeprefix("## ").strip() if lignes[0].startswith("## ") else f"section_{index}"
        chunks.append({
            "id": f"{doc['reference']}_titre_{index}",
            "texte": section,
            "meta": {
                "source": doc["source"],
                "reference": doc["reference"],
                "date": doc["date"],
                "site": doc.get("site"),
                "ligne": doc.get("ligne"),
                "section": titre,
                "strategie": "titre",
            },
        })

    return chunks


def chunk_taille_fixe(doc: dict, taille: int = 500, overlap: int = 80) -> list[dict]:
    texte = doc["texte"]
    if overlap >= taille:
        raise ValueError("overlap doit être strictement inférieur à taille")

    chunks = []
    pas = taille - overlap

    for index, start in enumerate(range(0, len(texte), pas), start=1):
        extrait = texte[start:start + taille].strip()
        if not extrait:
            continue

        chunks.append({
            "id": f"{doc['reference']}_fixe_{index}",
            "texte": extrait,
            "meta": {
                "source": doc["source"],
                "reference": doc["reference"],
                "date": doc["date"],
                "site": doc.get("site"),
                "ligne": doc.get("ligne"),
                "section": f"car_{start}_{start + len(extrait)}",
                "strategie": "fixe",
            },
        })

        if start + taille >= len(texte):
            break

    return chunks


chunks_titre = [chunk for doc in docs for chunk in chunk_par_titre(doc)]
chunks_fixes = [chunk for doc in docs for chunk in chunk_taille_fixe(doc)]

print(f"Chunks par titre : {len(chunks_titre)}")
print(f"Chunks par taille fixe : {len(chunks_fixes)}")

print("\nExemple chunk par titre :")
display(chunks_titre[0])

print("\nExemple chunk taille fixe :")
display(chunks_fixes[0])

print("\nChoix retenu pour la suite : chunk_par_titre")
print("Justification : les rapports sont courts, déjà structurés avec des sections Markdown, et la découpe par titre préserve mieux le sens métier qu'un découpage arbitraire en fenêtres fixes.")

Chunks par titre : 10
Chunks par taille fixe : 5

Exemple chunk par titre :


{'id': 'RT-2026-001_titre_2',
 'texte': "## Dérive thermique ligne 1\n\nSynthèse hebdomadaire des alertes supervision.\n\nTempérature moyenne relevée à 193 °C sur 6 h, au-dessus du seuil de consigne (180 °C). Cause probable : encrassement de l'échangeur. Intervention de nettoyage planifiée, retour à la normale sous 48 h.",
 'meta': {'source': 'RT-2026-001_Roubaix_ligne1.md',
  'reference': 'RT-2026-001',
  'date': '2026-04-01',
  'site': 'Roubaix',
  'ligne': '1',
  'section': 'Dérive thermique ligne 1',
  'strategie': 'titre'}}


Exemple chunk taille fixe :


{'id': 'RT-2026-001_fixe_1',
 'texte': "## Dérive thermique ligne 1\n\nSynthèse hebdomadaire des alertes supervision.\n\nTempérature moyenne relevée à 193 °C sur 6 h, au-dessus du seuil de consigne (180 °C). Cause probable : encrassement de l'échangeur. Intervention de nettoyage planifiée, retour à la normale sous 48 h.\n\n## Suite donnée\n\n- Responsable : EMP-2424\n- Statut : en cours\n- Lien supervision : voir `capteurs_iot.csv` (2026-04-01, site Roubaix, line_id 1).",
 'meta': {'source': 'RT-2026-001_Roubaix_ligne1.md',
  'reference': 'RT-2026-001',
  'date': '2026-04-01',
  'site': 'Roubaix',
  'ligne': '1',
  'section': 'car_0_424',
  'strategie': 'fixe'}}


Choix retenu pour la suite : chunk_par_titre
Justification : les rapports sont courts, déjà structurés avec des sections Markdown, et la découpe par titre préserve mieux le sens métier qu'un découpage arbitraire en fenêtres fixes.


## 4. Embeddings + indexation ChromaDB — **TODO**

Encode chaque chunk avec `SentenceTransformer(EMBED_MODEL)` (local), puis range
les chunks dans une collection ChromaDB **persistante**.

> Pense à **vider** la collection si elle existe déjà (idempotence du notebook).

In [28]:
import chromadb
from sentence_transformers import SentenceTransformer

chunks_retenus = chunks_titre

modele = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings = modele.encode(
    [chunk["texte"] for chunk in chunks_retenus]
).tolist()

client = chromadb.PersistentClient(path="./chroma_acerox")

try:
    client.delete_collection("rapports_acerox")
except Exception:
    pass

col = client.create_collection("rapports_acerox")

col.add(
    ids=[chunk["id"] for chunk in chunks_retenus],
    documents=[chunk["texte"] for chunk in chunks_retenus],
    embeddings=embeddings,
    metadatas=[chunk["meta"] for chunk in chunks_retenus],
)

print(f"Collection indexée : {col.count()} chunks")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Collection indexée : 10 chunks


## 5. Interroger par similarité — **TODO**

In [29]:
def interroger(question: str, k: int = 3) -> None:
    question_embedding = modele.encode([question]).tolist()

    resultats = col.query(
        query_embeddings=question_embedding,
        n_results=k,
    )

    documents = resultats.get("documents", [[]])[0]
    metadatas = resultats.get("metadatas", [[]])[0]
    distances = resultats.get("distances", [[]])[0]

    print(f"Question : {question}")
    print(f"Top {len(documents)} résultats :\n")

    for index, (document, metadata, distance) in enumerate(zip(documents, metadatas, distances), start=1):
        print(f"Résultat {index}")
        print(f"Distance : {distance:.4f}")
        print(
            f"Source : {metadata.get('source')} | Référence : {metadata.get('reference')} | "
            f"Date : {metadata.get('date')} | Section : {metadata.get('section')}"
        )
        print(document)
        print("\n" + "-" * 80 + "\n")


interroger("Pourquoi des défauts sur la ligne de laminage ?")

Question : Pourquoi des défauts sur la ligne de laminage ?
Top 3 résultats :

Résultat 1
Distance : 10.6721
Source : RT-2026-004_Lyon_ligne4.md | Référence : RT-2026-004 | Date : 2026-04-18 | Section : Défaut qualité lot L6925
## Défaut qualité lot L6925

Compte rendu d'intervention de maintenance corrective.

Taux de rebut de 5.7 % sur le lot L6925 (cible < 2 %). Corrélation avec la dérive thermique du 2026-04-18. Lot mis en quarantaine, analyse métallurgique demandée.

--------------------------------------------------------------------------------

Résultat 2
Distance : 10.9681
Source : RT-2026-005_Lyon_ligne2.md | Référence : RT-2026-005 | Date : 2026-04-28 | Section : Incident capteur S02
## Incident capteur S02

Note technique transmise au service industrialisation.

Trous de données (65 relevés manquants) sur le S02 entre le 2026-04-28 et le lendemain. Cause : perte de connexion réseau du concentrateur. Données interpolées non utilisées pour le scoring.

------------------------

## ✅ Vérification (checklist)

- [ ] J'ai chargé les 5 rapports + leurs métadonnées (loader fait main)
- [ ] J'ai comparé 2 stratégies de **segmentation** et justifié mon choix
- [ ] J'ai indexé les chunks dans ChromaDB avec des embeddings **locaux**
- [ ] Une requête en langage naturel récupère les bons passages
- [ ] J'ai vérifié que le **modèle d'embedding est adapté à la langue du corpus** (les rapports sont en français)
- [ ] J'ai écrit 3 lignes (dans ma note) sur **relationnel vs vectoriel**
- [ ] Je sais où s'arrête la **préparation** vs le **RAG complet** (M7-B2)

## ⭐ Pour aller plus loin
- Filtrer par métadonnée (`where={...}`) pour ne chercher que les rapports récents.
- Brancher un LLM local (Ollama) sur les chunks — mais : est-ce justifié ici ?